<a href="https://colab.research.google.com/github/kjahan/nlp_tips_and_trick/blob/main/notebooks/smart_paraphrasing_with_constrained_beam_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## We demo how to keep some words in paraphrasing intact

Use cases: copywriting and SEO!

https://towardsdatascience.com/smart-paraphrasing-using-constrained-beam-search-in-nlp-9af6fd046e5c


https://huggingface.co/blog/constrained-beam-search

In [9]:
!pip install -q sentencepiece
!pip install -q transformers==4.18.0

In [11]:
import torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("ramsrigouthamg/t5-large-paraphraser-diverse-high-quality")
tokenizer = AutoTokenizer.from_pretrained("ramsrigouthamg/t5-large-paraphraser-diverse-high-quality")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("device ",device)
model = model.to(device)

device  cpu


In [3]:
# Copy writing example - Use a given word in paraphrasing

context = "Supercharge your data science skills by building real world projects."

text = "paraphrase: "+context + " </s>"

input_ids = tokenizer(text, max_length=128, padding=True, return_tensors="pt").input_ids
input_ids = input_ids.to(device)

# Beam search

outputs = model.generate(
    input_ids,
    num_beams=10,
    num_return_sequences=5,
    max_length=128,
    early_stopping=True,
    no_repeat_ngram_size=1,
    remove_invalid_values=True,
)

print ("\nNormal beam search\n")
print ("Original: ",context)
for beam_output in outputs:
    sent = tokenizer.decode(beam_output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print (sent)

/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:2280: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  "`max_length` is ignored when `padding`=`True` and there is no truncation strategy. "



Normal beam search

Original:  Supercharge your data science skills by building real world projects.
paraphrasedoutput: By implementing real world projects, you can improve your data science skills.
paraphrasedoutput: By implementing real world projects, you can boost your data science skills.
paraphrasedoutput: By implementing real world projects, you can enhance your data science skills.
paraphrasedoutput: Develop your data science skills by implementing real world projects.
paraphrasedoutput: By implementing real world projects, you can raise your data science skills.


In [4]:
# Constrained Beam search
force_words = ["improve"]
force_words_ids = tokenizer(force_words, add_special_tokens=False).input_ids

outputs = model.generate(
    input_ids,
    force_words_ids=force_words_ids,
    max_length=128,
    early_stopping=True,
    num_beams=10,
    num_return_sequences=5,
    no_repeat_ngram_size=1,
    remove_invalid_values=True,
)

print ("\nConstrained beam search\n")
print ("Original: ",context)
for beam_output in outputs:
    sent = tokenizer.decode(beam_output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print (sent)


Constrained beam search

Original:  Supercharge your data science skills by building real world projects.
paraphrasedoutput: By implementing real world projects, you can improve your data science skills.
paraphrasedoutput: By executing real world projects, you can improve your data science skills.
paraphrasedoutput: Build real world projects to improve your data science skills.
paraphrasedoutput: Real world projects can help you improve your data science skills.
paraphrasedoutput: By implementing real world projects, you can improve your data science skills.


In [5]:
# Constrained Beam search
force_words = ["upgrade"]
force_words_ids = tokenizer(force_words, add_special_tokens=False).input_ids

outputs = model.generate(
    input_ids,
    force_words_ids=force_words_ids,
    max_length=128,
    early_stopping=True,
    num_beams=10,
    num_return_sequences=3,
    no_repeat_ngram_size=1,
    remove_invalid_values=True,
)

print ("\nConstrained beam search\n")
print ("Original: ",context)
for beam_output in outputs:
    sent = tokenizer.decode(beam_output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print (sent)


Constrained beam search

Original:  Supercharge your data science skills by building real world projects.
paraphrasedoutput: By implementing real world projects, you can upgrade your data science skills.
paraphrasedoutput: By implementing real world projects, you can improve your data science skills. The upgrade to Data Science by the end of this year will be greatly appreciated!
paraphrasedoutput: By implementing real world projects, you can improve your data science skills. The upgrade to Data Science by the end of this year will be greatly enhanced!


In [6]:
# Copy writing example - Use a given word in paraphrasing

context = "Apple products show a great taste of design and user focus. One good example is the iPhone."

text = "paraphrase: "+context + " </s>"

input_ids = tokenizer(text, max_length=128, padding=True, return_tensors="pt").input_ids
input_ids = input_ids.to(device)

# Beam search

outputs = model.generate(
    input_ids,
    num_beams=10,
    num_return_sequences=3,
    max_length=128,
    early_stopping=True,
    no_repeat_ngram_size=1,
    remove_invalid_values=True,
)

print ("\nNormal beam search\n")
print ("Original: ",context)
for beam_output in outputs:
    sent = tokenizer.decode(beam_output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print (sent)

/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:2280: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  "`max_length` is ignored when `padding`=`True` and there is no truncation strategy. "



Normal beam search

Original:  Apple products show a great taste of design and user focus. One good example is the iPhone.
paraphrasedoutput: Apple's iPhone is a good example.
paraphrasedoutput: Apple products are a good example of both form and customer attention.
paraphrasedoutput: Apple products are a good example of both design and user focus.


In [7]:
# Constrained Beam search
force_words = ["iPhone"]
force_words_ids = tokenizer(force_words, add_special_tokens=False).input_ids

outputs = model.generate(
    input_ids,
    force_words_ids=force_words_ids,
    max_length=128,
    early_stopping=True,
    num_beams=10,
    num_return_sequences=3,
    no_repeat_ngram_size=1,
    remove_invalid_values=True,
)

print ("\nConstrained beam search\n")
print ("Original: ",context)
for beam_output in outputs:
    sent = tokenizer.decode(beam_output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print (sent)


Constrained beam search

Original:  Apple products show a great taste of design and user focus. One good example is the iPhone.
paraphrasedoutput: The iPhone is a good example of design and user focus.
paraphrasedoutput: The iPhone is a good example of both design and user focus.
paraphrasedoutput: The iPhone is a good example.


In [8]:
# Constrained Beam search
force_words = ["iPhone", "customers"]
force_words_ids = tokenizer(force_words, add_special_tokens=False).input_ids

outputs = model.generate(
    input_ids,
    force_words_ids=force_words_ids,
    max_length=128,
    early_stopping=True,
    num_beams=10,
    num_return_sequences=3,
    no_repeat_ngram_size=1,
    remove_invalid_values=True,
)

print ("\nConstrained beam search\n")
print ("Original: ",context)
for beam_output in outputs:
    sent = tokenizer.decode(beam_output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print (sent)


Constrained beam search

Original:  Apple products show a great taste of design and user focus. One good example is the iPhone.
paraphrasedoutput: Apple's iPhone is a good example of both form and customers.
paraphrasedoutput: Apple's iPhone is a good example of both form and customer attention. The customers are always on alert when it comes to apple products, as shown by this one in the screenshot above (right).
paraphrasedoutput: Apple's iPhone is a good example of both form and customer attention. The customers are always on alert when it comes to apple products, as shown by this one in the image above (right).


## Amazon example

https://www.amazon.com/The-Mom-Test-Rob-Fitzpatrick-audiobook/dp/B07RJZKZ7F

In [13]:
# Copy writing example - Use a given word in paraphrasing

context = """
They say you shouldn't ask your mom whether your business is a good idea, because she loves you and will lie to you.
"""

text = "paraphrase: "+context + " </s>"

input_ids = tokenizer(text, max_length=128, padding=True, return_tensors="pt").input_ids
input_ids = input_ids.to(device)

# Beam search

outputs = model.generate(
    input_ids,
    num_beams=10,
    num_return_sequences=3,
    max_length=128,
    early_stopping=True,
    no_repeat_ngram_size=1,
    remove_invalid_values=True,
)

print ("\nNormal beam search\n")
print ("Original: ",context)
for beam_output in outputs:
    sent = tokenizer.decode(beam_output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print (sent)


Normal beam search

Original:  
They say you shouldn't ask your mom whether your business is a good idea, because she loves you and will lie to you.

paraphrasedoutput: According to them, you shouldn't ask your mother whether starting a business would be beneficial because she cares about her and will lie in the future.
paraphrasedoutput: According to them, you shouldn't ask your mother whether starting a new company would be beneficial because she cares about her and will lie in the future.
paraphrasedoutput: According to them, you shouldn't ask your mother whether starting a business would be beneficial because she cares about her and will lie in the mail.


In [14]:
# Constrained Beam search
force_words = ["mom"]
force_words_ids = tokenizer(force_words, add_special_tokens=False).input_ids

outputs = model.generate(
    input_ids,
    force_words_ids=force_words_ids,
    max_length=128,
    early_stopping=True,
    num_beams=10,
    num_return_sequences=3,
    no_repeat_ngram_size=1,
    remove_invalid_values=True,
)

print ("\nConstrained beam search\n")
print ("Original: ",context)
for beam_output in outputs:
    sent = tokenizer.decode(beam_output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print (sent)


Constrained beam search

Original:  
They say you shouldn't ask your mom whether your business is a good idea, because she loves you and will lie to you.

paraphrasedoutput: According to them, you shouldn't ask your mother if her company is legitimate because she loves moms and will lie in the future.
paraphrasedoutput: According to them, you shouldn't ask your mother if her company is worthwhile because she loves moms and will lie in the future.
paraphrasedoutput: According to them, you shouldn't ask your mother if her company is legitimate because she loves moms and will lie in the mail.


## Nvidia GPU example

https://www.amazon.com/PNY-GeForce-Gaming-Epic-X-Graphics/dp/B096W957CP/ref=sr_1_1_sspa?crid=3T834TF5NCSQ5&keywords=gpu&qid=1664750723&qu=eyJxc2MiOiI3LjY2IiwicXNhIjoiNy4wMyIsInFzcCI6IjYuOTAifQ%3D%3D&sprefix=gpu%2Caps%2C179&sr=8-1-spons&psc=1&spLa=ZW5jcnlwdGVkUXVhbGlmaWVyPUEzTUlGUDIxWkRXRlFVJmVuY3J5cHRlZElkPUEwODc0NTk2MjlMQ1MySFgwRVkxTyZlbmNyeXB0ZWRBZElkPUEwMzUwMTk3MlJaM0VVN0pHRk45WSZ3aWRnZXROYW1lPXNwX2F0ZiZhY3Rpb249Y2xpY2tSZWRpcmVjdCZkb05vdExvZ0NsaWNrPXRydWU=

`NVIDIA Ampere architecture, with 1575MHz core clock and 1770MHz boost clock speeds to help meet the needs of demanding games.`

`8GB GDDR6X (256-bit) on-board memory, plus 6144 CUDA processing cores and up to 608GB/sec of memory bandwidth provide the memory needed to create striking visual realism.`

`PCI Express 4.0 interface - Offers compatibility with a range of systems. Also includes DisplayPort and HDMI outputs for expanded connectivity.`

`NVIDIA GeForce Experience - Capture and share videos, screenshots, and livestreams with friends. Keep your drivers up to date and optimize your game settings. It's the essential companion to your GeForce graphics card.`

In [15]:
# Copy writing example - Use a given word in paraphrasing

context = """
NVIDIA Ampere architecture, with 1575MHz core clock and 1770MHz boost clock speeds to help meet the needs of demanding games.
"""

text = "paraphrase: "+context + " </s>"

input_ids = tokenizer(text, max_length=128, padding=True, return_tensors="pt").input_ids
input_ids = input_ids.to(device)

# Beam search

outputs = model.generate(
    input_ids,
    num_beams=10,
    num_return_sequences=3,
    max_length=128,
    early_stopping=True,
    no_repeat_ngram_size=1,
    remove_invalid_values=True,
)

print ("\nNormal beam search\n")
print ("Original: ",context)
for beam_output in outputs:
    sent = tokenizer.decode(beam_output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print (sent)


Normal beam search

Original:  
NVIDIA Ampere architecture, with 1575MHz core clock and 1770MHz boost clock speeds to help meet the needs of demanding games.

paraphrasedoutput: To help satisfy the needs of demanding games, NVIDIA's Ampere architecture with a 1575MHz core clock and 1770 percent boost speed.
paraphrasedoutput: To help satisfy the needs of demanding games, NVIDIA's Ampere architecture with a 1575MHz core clock and 1770 percent increase speed.
paraphrasedoutput: To help satisfy the needs of demanding games, NVIDIA's Ampere architecture with a 1575MHz core clock and 1770 percent boostclock speeds.


In [16]:
# Copy writing example - Use a given word in paraphrasing

context = """
NVIDIA GeForce Experience - Capture and share videos, screenshots, and livestreams with friends. Keep your drivers up to date and optimize your game settings. 
It's the essential companion to your GeForce graphics card.
"""

text = "paraphrase: "+context + " </s>"

input_ids = tokenizer(text, max_length=128, padding=True, return_tensors="pt").input_ids
input_ids = input_ids.to(device)

# Beam search

outputs = model.generate(
    input_ids,
    num_beams=10,
    num_return_sequences=3,
    max_length=128,
    early_stopping=True,
    no_repeat_ngram_size=1,
    remove_invalid_values=True,
)

print ("\nNormal beam search\n")
print ("Original: ",context)
for beam_output in outputs:
    sent = tokenizer.decode(beam_output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print (sent)


Normal beam search

Original:  
NVIDIA GeForce Experience - Capture and share videos, screenshots, and livestreams with friends. Keep your drivers up to date and optimize your game settings. It's the essential companion to your GeForce graphics card.

paraphrasedoutput: The NVIDIA GeForce Experience - Capture and Share videos, screenshots of your games or livestreaming with friends. It is the most useful companion to you when it comes on!
paraphrasedoutput: The NVIDIA GeForce Experience - Capture and Post videos, screenshots of your games or livestreaming with friends. It is the most useful companion to you when it comes on!
paraphrasedoutput: The NVIDIA GeForce Experience - Capture and Share videos, screenshots of your games or livestreaming with friends. It is the most useful companion to you when it comes from an Android phone!


In [18]:
# Constrained Beam search
force_words = ["GeForce", "graphics", "card", "livestreaming"]
force_words_ids = tokenizer(force_words, add_special_tokens=False).input_ids

outputs = model.generate(
    input_ids,
    force_words_ids=force_words_ids,
    max_length=128,
    early_stopping=True,
    num_beams=10,
    num_return_sequences=3,
    no_repeat_ngram_size=1,
    remove_invalid_values=True,
)

print ("\nConstrained beam search\n")
print ("Original: ",context)
for beam_output in outputs:
    sent = tokenizer.decode(beam_output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print (sent)


Constrained beam search

Original:  
NVIDIA GeForce Experience - Capture and share videos, screenshots, and livestreams with friends. Keep your drivers up to date and optimize your game settings. It's the essential companion to your GeForce graphics card.

par graphics card GeForce Experience - Capture and livestream videos livestreaming with friends.
paraphrasedoutput GeForce Experience - Capture and livestream videos livestreaming with colleagues. It's the ultimate companion to your graphics card, which is also known as "the best friend."
paraphrasedoutput GeForce Experience - Capture and livestream videos livestreaming with friends. It's the ultimate companion to your graphics card, which is also known as "the best friend."
